In [1]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


def train_student_ticket_ai(data, confidence_threshold=0.60):
    """
    Trains an AI model to classify higher-education student support tickets.

    Features:
    - Text preprocessing using TF-IDF
    - Multi-class classification
    - Cross-validation accuracy
    - Confidence score prediction
    - Low-confidence escalation flag
    - Top keywords per category
    """

    df = pd.DataFrame(data)

    X = df["ticket_text"]
    y = df["category"]

    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=1
        )),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ))
    ])

    scores = cross_val_score(model, X, y, cv=3)

    model.fit(X, y)

    def predict_ticket(ticket_text):
        probabilities = model.predict_proba([ticket_text])[0]
        categories = model.classes_

        best_index = np.argmax(probabilities)
        predicted_category = categories[best_index]
        confidence = probabilities[best_index]

        if confidence < confidence_threshold:
            action = "Escalate to human advisor"
        else:
            action = "Auto-route to department"

        top_3 = sorted(
            zip(categories, probabilities),
            key=lambda x: x[1],
            reverse=True
        )[:3]

        return {
            "ticket": ticket_text,
            "predicted_category": predicted_category,
            "confidence": round(confidence, 3),
            "recommended_action": action,
            "top_3_predictions": top_3
        }

    def show_top_keywords(n=5):
        tfidf = model.named_steps["tfidf"]
        classifier = model.named_steps["classifier"]

        feature_names = np.array(tfidf.get_feature_names_out())

        keywords = {}

        for i, category in enumerate(classifier.classes_):
            top_indices = classifier.coef_[i].argsort()[-n:][::-1]
            keywords[category] = feature_names[top_indices].tolist()

        return keywords

    return {
        "model": model,
        "cross_validation_accuracy": round(scores.mean(), 3),
        "predict_ticket": predict_ticket,
        "top_keywords": show_top_keywords
    }


data = {
    "ticket_text": [
        "I cannot enroll in my data science course because Quest shows an error",
        "My tuition payment was made but the balance still appears unpaid",
        "I need academic advice about choosing electives for next term",
        "I want to know if I am eligible to graduate this spring",
        "I cannot log into my student portal or reset my password",
        "Can I defer my admission offer to the next intake?",
        "My OSAP funding has not been applied to my student account",
        "I need help understanding my degree requirements",
        "The learning management system is not loading my course page",
        "I want to change my application program before the deadline",
        "I dropped a course but I am still being charged tuition",
        "Can an advisor help me plan my final semester?",
        "Where do I submit my intent to graduate form?",
        "My two-factor authentication is not working",
        "When will I receive a decision on my graduate application?"
    ],
    "category": [
        "course_registration",
        "financial_aid",
        "academic_advising",
        "graduation",
        "technical_support",
        "admissions",
        "financial_aid",
        "academic_advising",
        "technical_support",
        "admissions",
        "financial_aid",
        "academic_advising",
        "graduation",
        "technical_support",
        "admissions"
    ]
}


ai_system = train_student_ticket_ai(data)

print("Cross-validation accuracy:", ai_system["cross_validation_accuracy"])

test_ticket = "I cannot access my student account and my password reset link is expired"

result = ai_system["predict_ticket"](test_ticket)

print("\nPrediction Result:")
for key, value in result.items():
    print(f"{key}: {value}")

print("\nTop Keywords by Category:")
print(ai_system["top_keywords"](n=5))

Cross-validation accuracy: 0.6

Prediction Result:
ticket: I cannot access my student account and my password reset link is expired
predicted_category: technical_support
confidence: 0.213
recommended_action: Escalate to human advisor
top_3_predictions: [('technical_support', 0.21344617934373963), ('financial_aid', 0.21108879023052055), ('academic_advising', 0.1568472990521087)]

Top Keywords by Category:
{'academic_advising': ['help', 'need', 'help plan', 'advisor help', 'semester'], 'admissions': ['application', 'receive', 'receive decision', 'graduate application', 'decision'], 'course_registration': ['shows', 'shows error', 'error', 'enroll data', 'enroll'], 'financial_aid': ['tuition', 'charged', 'dropped course', 'dropped', 'course charged'], 'graduation': ['graduate', 'submit intent', 'intent graduate', 'intent', 'form'], 'technical_support': ['working', 'authentication', 'factor authentication', 'authentication working', 'factor']}


C:\Users\fedab\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
